In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)



### Implemented Manual EDA

In [ ]:
DATA_PATH = "../data/predictive_maintenance.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
print(df.columns.tolist())
df.info()

In [ ]:
df.describe()

In [ ]:
print("Target counts:\n", df["Target"].value_counts())
print("\nTarget rate (mean):", round(df["Target"].mean(), 4))
print("\nTarget %:\n", (df["Target"].value_counts(normalize=True) * 100).round(2))

In [ ]:
df["Failure Type"].value_counts(dropna=False).head(10)

drop_cols = ["UDI", "Product ID", "Failure Type"]
df_model = df.drop(columns=drop_cols)

print("New shape after dropping leakage/IDs:", df_model.shape)
df_model.head()

In [ ]:
y = df_model["Target"].astype(int)
X = df_model.drop(columns=["Target"])

# Encode machine Type 
X = pd.get_dummies(X, columns=["Type"], drop_first=True)

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()

### IMPLEMENTED TRAINING AND TESTING FOR MODEL

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train failure rate:", round(y_train.mean(), 4))
print("Test failure rate:", round(y_test.mean(), 4))

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

lr_acc = accuracy_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)

print("Logistic Regression")
print("Accuracy:", round(lr_acc, 4))
print("F1:", round(lr_f1, 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, lr_pred))

### Baseline Model Interpretation

Logistic Regression establishes a simple interpretable baseline.
Due to class imbalance (~3.4% failures), F1-score is used instead
of accuracy to evaluate performance.

The model prioritizes detecting failures while accepting some
false positives, which aligns with predictive maintenance goals.

### XGBOOST MODEL IMPLEMENTATION

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric='logloss'
)

xgb.fit(X_train_scaled, y_train)
xgb_pred = xgb.predict(X_test_scaled)

xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)

print("XGBoost")
print("Accuracy:", round(xgb_acc, 4))
print("F1:", round(xgb_f1, 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, xgb_pred))

plt.figure(figsize=(10, 6))
plt.barh(X.columns, xgb.feature_importances_)
plt.title("XGBoost Feature Importances (Key Drivers of Failure)")
plt.xlabel("Importance Score")
plt.show()

### BUILDING AI AGENT


In [ ]:

import os
import getpass
import numpy as np
import pandas as pd

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# API key
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")



# storing notebook vars
STATE = {
    "df": df,
    "X_cols": list(X.columns),
    "lr_model": lr,
    "xgb_model": xgb,
    "X_test": X_test,
    "X_test_scaled": X_test_scaled, 
    "y_test": y_test
}



# TOOL 1: EDA SUMMARY

@tool
def eda_tool(query: str) -> str:
    """
    Use this tool for ANY dataset exploration or EDA questions.

    Use when the user asks about:
    - dataset summary
    - number of rows or columns
    - class imbalance
    - Target distribution
    - missing values
    - correlations with Target
    - exploratory data analysis

    Example questions:
    "Summarize the dataset"
    "Is the Target imbalanced?"
    "Show dataset statistics"
    "What features correlate with Target?"

    This tool returns:
    - dataset shape
    - Target class distribution
    - missing values
    - top correlations with Target
    """

    df_local = STATE["df"]

    report = []

    report.append(f"Dataset shape: {df_local.shape[0]} rows, {df_local.shape[1]} columns\n")

    target_counts = df_local["Target"].value_counts()
    target_pct = (target_counts / len(df_local) * 100).round(2)

    report.append("Target distribution:")
    report.append(pd.DataFrame({
        "count": target_counts,
        "percent": target_pct
    }).to_string() + "\n")

    missing = df_local.isna().sum()
    missing = missing[missing > 0]

    if len(missing) == 0:
        report.append("Missing values: None\n")
    else:
        report.append("Missing values:")
        report.append(missing.to_string() + "\n")

    numeric_cols = df_local.select_dtypes(include=[np.number]).columns.tolist()
    if "Target" in numeric_cols:
        numeric_cols.remove("Target")

    corr = df_local[numeric_cols + ["Target"]].corr(numeric_only=True)["Target"].drop("Target")
    corr = corr.reindex(corr.abs().sort_values(ascending=False).index)

    report.append("Top correlations with Target:")
    report.append(corr.head(5).round(4).to_string())

    return "\n".join(report)


# TOOL 2: MODEL COMPARISON 
@tool
def performance_comparison_tool(query: str) -> str:
    """
    Use this tool for ANY machine learning model evaluation questions.

    Use when the user asks about:
    - model performance
    - comparing Logistic Regression vs XGBoost
    - accuracy, precision, recall, or F1
    - confusion matrices
    - which model performs better
    - model evaluation

    Example questions:
    "Compare Logistic Regression and XGBoost"
    "Which model is better?"
    "Show confusion matrices"
    "Evaluate model performance"

    This tool returns:
    - Accuracy
    - Precision
    - Recall
    - F1 Score
    - Confusion matrices for both models
    - A conclusion about which model performs better
    """

    y_test = np.ravel(STATE["y_test"])

    lr_model = STATE["lr_model"]
    xgb_model = STATE["xgb_model"]

    
    if "X_test_scaled" in STATE:
        X_eval = STATE["X_test_scaled"]
        scaling_note = "Using X_test_scaled from STATE."
    elif "scaler" in STATE:
        X_eval = STATE["scaler"].transform(STATE["X_test"])
        scaling_note = "Scaled X_test using STATE['scaler']."
    else:
        
        X_eval = STATE["X_test"]
        scaling_note = "WARNING: Using raw X_test (no scaler found). Metrics may be wrong."

    lr_pred = lr_model.predict(X_eval)
    xgb_pred = xgb_model.predict(X_eval)

    def metrics(pred):
        return (
            accuracy_score(y_test, pred),
            precision_score(y_test, pred, zero_division=0),
            recall_score(y_test, pred, zero_division=0),
            f1_score(y_test, pred, zero_division=0),
            confusion_matrix(y_test, pred)
        )

    lr_acc, lr_prec, lr_rec, lr_f1, lr_cm = metrics(lr_pred)
    xgb_acc, xgb_prec, xgb_rec, xgb_f1, xgb_cm = metrics(xgb_pred)

    report = []

    report.append("MODEL PERFORMANCE COMPARISON\n")
    report.append(f"Scaling check: {scaling_note}\n")

    report.append("Logistic Regression:")
    report.append(f"Accuracy:  {lr_acc:.4f}")
    report.append(f"Precision: {lr_prec:.4f}")
    report.append(f"Recall:    {lr_rec:.4f}")
    report.append(f"F1 Score:  {lr_f1:.4f}")
    report.append(str(lr_cm) + "\n")

    report.append("XGBoost:")
    report.append(f"Accuracy:  {xgb_acc:.4f}")
    report.append(f"Precision: {xgb_prec:.4f}")
    report.append(f"Recall:    {xgb_rec:.4f}")
    report.append(f"F1 Score:  {xgb_f1:.4f}")
    report.append(str(xgb_cm) + "\n")

    report.append("Conclusion:")

    if xgb_f1 > lr_f1:
        report.append("XGBoost performs better overall based on F1.")
    else:
        report.append("Logistic Regression performs better overall based on F1.")

    report.append("For failure prediction, Recall and F1 are usually more important than Accuracy.")

    return "\n".join(report)



# TOOL 3: FEATURE DRIVERS

@tool
def action_recommendations_tool(sensor_focus: str) -> str:
    """Use this tool for ANY feature importance or operational insight questions.

    Use when the user asks about:
    - drivers of machine failure
    - feature importance
    - why failures happen
    - maintenance recommendations
    - what fleet managers should monitor
    - sensor analysis such as torque or speed

    Example questions:
    "What causes machine failures?"
    "What features drive failure predictions?"
    "What should mechanics monitor?"
    "Focus on torque"
    "Focus on speed"

    This tool returns:
    - top 3 failure drivers from XGBoost feature importance
    - operational maintenance recommendations"""

    model = STATE["xgb_model"]
    features = STATE["X_cols"]

    importances = model.feature_importances_

    imp_df = pd.DataFrame({
        "Feature": features,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)

    top1 = imp_df.iloc[0]["Feature"]
    top2 = imp_df.iloc[1]["Feature"]
    top3 = imp_df.iloc[2]["Feature"]

    report = []

    report.append("Top drivers of machine failure:")
    report.append(f"1. {top1}")
    report.append(f"2. {top2}")
    report.append(f"3. {top3}\n")

    report.append("Operational recommendations:")
    report.append(f"- Monitor {top1} closely across the fleet.")
    report.append(f"- Add checks for {top2} during scheduled maintenance.")
    report.append(f"- Track {top3} trends across trucks.\n")

    if "torque" in sensor_focus.lower():
        report.append("Torque spikes may indicate drivetrain stress.")
    elif "speed" in sensor_focus.lower():
        report.append("Speed anomalies may indicate engine control issues.")

    return "\n".join(report)



# BUILDING AGENT

tools = [eda_tool, performance_comparison_tool, action_recommendations_tool]

llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

agent_executor = create_react_agent(llm, tools)



# INTERACTIVE DEMO LOOP

print("\n" + "="*50)
print("FLEET AI AGENT")
print("Type 'quit' to exit.")
print("="*50 + "\n")

while True:

    user_question = input("\nFleet Manager: ")

    if user_question.lower() in ["quit", "exit", "stop"]:
        print("Agent Stopped")
        break

    print("prompt: ", user_question)

    print("\n[AI Agent is thinking]")

    final_state = agent_executor.invoke({
        "messages": [("user", user_question)]
    })

    print("\nAI RESPONSE:")
    print(final_state["messages"][-1].content)
    print("-"*50)